In [2]:
"""
Rank ENSO reconstructions within each proxy type by a globally normalised weighted composite score

Metrics and weights:
  1. frac_valid            (w=0.30) — temporal coverage
  2. interannual_var_ratio (w=0.35) — variance in 2–7 yr ENSO band / total variance
  3. var_stability         (w=0.25) — temporal homogeneity of variance (50-yr rolling)
  4. median_corr_within    (w=0.10) — within-paper-group coherence
"""

import warnings
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt


# USER SETTINGS

FILE_PATH = "/home/563/ft3359/ENSO_Records_all.csv"
YEAR_COL  = "Years"

CORAL_COLUMNS = [
    "Wilson et al. (2010) Nino34 COACPR",
    "Wilson et al. (2010) Nino34 COAPCR",
    "Wilson et al. 2010 Nino34 zscoreCOA",
    "Tierney et al. (2015) East",
    "Tierney et al. (2015) West",
    "Freund et al. (2019) NCT DJF",
    "Freund et al. (2019) NWP DJF",
    "Freund et al. (2019) Nino4 DJF",
    "Freund et al. (2019) Nino3 DJF",
    "Zhu et al. (2022) Corals Li13b6",
    "Zhu et al. (2022) Corals",
    "Zhu et al. (2022) Li13b6.",
    "Zhu et al. (2022) Ocn2kCorals Li13b6",
]

TREE_RING_COLUMNS = [
    "Stahle et al. (1993)",
    "Stahle et al. (1998)",
    "Li et al. (2011) NADA PC1",
    "Li et al. (2013) Tree-rings PC1",
    "DArrigo et al. (2005) Nino3",
    "Torbenson et al. (2019) eMEIall",
    "Torbenson et al. (2019) eMEIstable",
]

MULTI_PROXY_COLUMNS = [
    "Datwyler et al. (2020) ENSO DJF",
    "Datwyler et al. (2019) FullPeriod PC1",
    "Mann et al. (2000)",
    "Wilson et al. (2010) Nino34 TelecPCR TEL",
    "Wilson et al. (2010) Nino34 zscoreTEL",
    "McGregor et al. (2010)",
    "Braganza et al. (2009) Multiproxy R5",
    "Braganza et al. (2009) Multiproxy R8",
    "Falster et al.(2023)",
    "Liu et al. (2024) PCR",
    "Liu et al. (2024) DD",
    "Geay et al. (2013) Nino3",
]

WEIGHTS = {
    "frac_valid":            0.30,
    "interannual_var_ratio": 0.35,
    "var_stability":         0.25,
    "median_corr_within":    0.10,
}

START_YEAR           = 850
END_YEAR             = 1849
MIN_OVERLAP          = 30
MIN_YEARS_SPECTRAL   = 50
VAR_STABILITY_WINDOW = 50


# INTERNAL REGISTRY

PROXY_REGISTRY = {
    "coral":       CORAL_COLUMNS,
    "tree-ring":   TREE_RING_COLUMNS,
    "multi-proxy": MULTI_PROXY_COLUMNS,
}

ALL_RECON_COLUMNS = CORAL_COLUMNS + TREE_RING_COLUMNS + MULTI_PROXY_COLUMNS
METRIC_KEYS = ["frac_valid", "interannual_var_ratio",
               "var_stability", "median_corr_within"]


# HELPERS

def detect_year_col(df, preferred="Years"):
    if preferred in df.columns:
        return preferred
    for c in ["Years", "Year", "YEAR", "years", "time", "Time", "date"]:
        if c in df.columns:
            return c
    raise ValueError("Cannot detect year column; set YEAR_COL manually.")


def clean_colname(c):
    return str(c).strip().strip("'").strip('"').strip()


def paper_key(colname):
    import re
    m = re.match(r"^(.+?)\(?\b(\d{4})\b\)?", colname)
    if m:
        return f"{m.group(1).strip().rstrip('.')} ({m.group(2)})"
    return "Other/Unknown"


def proxy_type_of(colname):
    for ptype, cols in PROXY_REGISTRY.items():
        if colname in cols:
            return ptype
    return "unknown"


# METRIC CALCULATIONS

def calc_frac_valid(s):
    return 0.0 if s.empty else float(s.notna().sum() / len(s))


def calc_interannual_var_ratio(s):
    valid = s.dropna()
    if len(valid) < MIN_YEARS_SPECTRAL:
        return np.nan
    s_interp = s.interpolate(method="linear", limit_direction="forward").dropna()
    n = len(s_interp)
    if n < MIN_YEARS_SPECTRAL:
        return np.nan

    b, a = butter(4, [1/7, 0.4999], btype="band", fs=1.0)
    if n <= 3 * max(len(a), len(b)):
        return np.nan
    try:
        filtered = filtfilt(b, a, s_interp.values)
    except ValueError:
        return np.nan
    total_var = float(np.var(s_interp.values, ddof=1))
    if total_var == 0:
        return np.nan
    return float(np.clip(np.var(filtered, ddof=1) / total_var, 0.0, 1.0))


def calc_var_stability(s, window=VAR_STABILITY_WINDOW):
    valid   = s.dropna()
    n_valid = len(valid)
    if n_valid < 2 * window:
        return np.nan
    rolling_var = s.rolling(window=window, min_periods=window // 2).var().dropna()
    n_windows   = len(rolling_var)
    if n_windows < 3:
        warnings.warn(
            f"var_stability: only {n_windows} rolling window(s) for series "
            f"with {n_valid} valid values (window={window}). "
            "Metric may be unreliable.", stacklevel=2,
        )
        if n_windows < 2:
            return np.nan
    mean_var = float(rolling_var.mean())
    if mean_var == 0:
        return np.nan
    cv = float(rolling_var.std(ddof=1) / mean_var)
    return float(1.0 - min(cv, 2.0) / 2.0)


def calc_median_corr_within(col, df_group):
    others = [c for c in df_group.columns if c != col]
    if not others:
        return np.nan
    s     = df_group[col].dropna()
    corrs = []
    for oc in others:
        o      = df_group[oc].dropna()
        shared = s.index.intersection(o.index)
        if len(shared) >= MIN_OVERLAP:
            corrs.append(float(np.corrcoef(s.loc[shared], o.loc[shared])[0, 1]))
    return float(np.median(corrs)) if corrs else np.nan


def compute_metrics(col, df_group):
    s = df_group[col]
    return {
        "series":                col,
        "proxy_type":            proxy_type_of(col),
        "paper":                 paper_key(col),
        "frac_valid":            calc_frac_valid(s),
        "interannual_var_ratio": calc_interannual_var_ratio(s),
        "var_stability":         calc_var_stability(s),
        "median_corr_within":    calc_median_corr_within(col, df_group),
    }


# GLOBAL NORMALISATION AND SCORING

def build_global_metrics(df_sub, groups):
    """
    Compute raw metrics for every series, then min-max normalise each
    metric GLOBALLY across all series so scores are cross-proxy comparable.
    """
    rows = []
    for gname, info in groups.items():
        df_grp = df_sub[info["cols"]]
        for col in info["cols"]:
            rows.append(compute_metrics(col, df_grp))

    metrics = pd.DataFrame(rows)

    # Global min-max normalisation
    for key in METRIC_KEYS:
        col = metrics[key].astype(float)
        mn, mx = col.min(skipna=True), col.max(skipna=True)
        if pd.isna(mn) or mn == mx:
            metrics[f"{key}_norm"] = 0.0
        else:
            metrics[f"{key}_norm"] = (col - mn) / (mx - mn)

    # Weighted composite (NaN normalised values treated as 0)
    metrics["composite_score"] = sum(
        WEIGHTS[k] * metrics[f"{k}_norm"].fillna(0.0)
        for k in METRIC_KEYS
    )

    return metrics


# RANKED OUTPUT

def print_proxy_rankings(metrics: pd.DataFrame):
    """
    Print one ranked table per proxy type, sorted by composite score.
    Scores are globally normalised so ranks are cross-proxy comparable.
    """
    display_cols = [
        "rank", "series", "paper",
        "frac_valid", "interannual_var_ratio",
        "var_stability", "median_corr_within",
        "composite_score",
    ]

    print(f"Weights: frac_valid={WEIGHTS['frac_valid']}  "
          f"interannual_var_ratio={WEIGHTS['interannual_var_ratio']}  "
          f"var_stability={WEIGHTS['var_stability']}  "
          f"median_corr_within={WEIGHTS['median_corr_within']}")
    print("Scores are globally min-max normalised across ALL series.\n")

    for ptype in ["coral", "tree-ring", "multi-proxy"]:
        subset = (
            metrics[metrics["proxy_type"] == ptype]
            .sort_values("composite_score", ascending=False)
            .copy()
            .reset_index(drop=True)
        )
        subset.insert(0, "rank", subset.index + 1)

        print(f"PROXY TYPE: {ptype.upper()}  ({len(subset)} series)")

        with pd.option_context("display.max_colwidth", 55,
                               "display.width", 220):
            print(subset[display_cols].to_string(
                index=False,
                float_format=lambda x: f"{x:.4f}",
            ))

        best = subset.iloc[0]
        print(f">> Top-ranked: {best['series']}"
              f"(composite={best['composite_score']:.4f})\n")


# MAIN

df = pd.read_csv(FILE_PATH)
df.columns = [clean_colname(c) for c in df.columns]
YEAR_COL = detect_year_col(df, preferred=YEAR_COL)

missing = [c for c in ALL_RECON_COLUMNS if c not in df.columns]
if missing:
    print("WARNING — columns not found in CSV:")
    for c in missing:
        print(f"  - {c}")

use_cols = [c for c in ALL_RECON_COLUMNS if c in df.columns]
if not use_cols:
    raise ValueError("None of the requested columns exist in the CSV.")

years = pd.to_numeric(df[YEAR_COL], errors="coerce")
mask  = np.isfinite(years)
if START_YEAR is not None:
    mask &= years >= START_YEAR
if END_YEAR is not None:
    mask &= years <= END_YEAR

df_sub    = df.loc[mask, [YEAR_COL] + use_cols].copy()
years_sub = pd.to_numeric(df_sub[YEAR_COL], errors="coerce")
for c in use_cols:
    df_sub[c] = pd.to_numeric(df_sub[c], errors="coerce")

# Group by paper, splitting where one paper spans two proxy types
groups: dict = {}
for col in use_cols:
    g     = paper_key(col)
    ptype = proxy_type_of(col)
    if g in groups and groups[g]["proxy_type"] != ptype:
        g = f"{g} [{ptype}]"
    if g not in groups:
        groups[g] = {"proxy_type": ptype, "cols": []}
    groups[g]["cols"].append(col)

# Compute globally normalised metrics and print rankings
metrics = build_global_metrics(df_sub, groups)
print_proxy_rankings(metrics)

Weights: frac_valid=0.3  interannual_var_ratio=0.35  var_stability=0.25  median_corr_within=0.1
Scores are globally min-max normalised across ALL series.

PROXY TYPE: CORAL  (13 series)
 rank                               series                paper  frac_valid  interannual_var_ratio  var_stability  median_corr_within  composite_score
    1            Zhu et al. (2022) Li13b6.     Zhu et al (2022)      0.7895                 0.5751         0.8468              0.8838           0.7009
    2 Zhu et al. (2022) Ocn2kCorals Li13b6     Zhu et al (2022)      0.7895                 0.4992         0.8626              0.8838           0.6776
    3      Zhu et al. (2022) Corals Li13b6     Zhu et al (2022)      0.7895                 0.5014         0.8606              0.8853           0.6771
    4   Wilson et al. (2010) Nino34 COAPCR  Wilson et al (2010)      0.2558                 0.5524         0.9044              0.9974           0.5644
    5   Wilson et al. (2010) Nino34 COACPR  Wilson et al (2